# Query the RAG database

This notebook only **reads** the Chroma database built in `populate_db_test.ipynb`. 

What a query does:
1. Embed the question with the same model used to build the database (currently `bge-m3`).
2. Find the `k` closest chunks in Chroma.
3. Paste those chunks into a prompt as context.
4. Ask the LLM to answer using only that context.
5. Show the answer and where it came from (file, page, section).

## Imports

In [ ]:
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from ollama import chat 

from pathlib import Path
import sys

# Notebooks live in notebooks/, so make src/ importable (config.py, get_embedding_function.py)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import DEFAULT_CORPUS, get_chroma_path
from get_embedding_function import get_embedding_function

## Settings

In [ ]:
CORPUS = DEFAULT_CORPUS    # must match the corpus built in the populate notebook
CHROMA_PATH = str(get_chroma_path(CORPUS))   # <project>/vector_stores/chroma_<corpus>
K = 5                      # chunks to retrieve
LLM_MODEL = "qwen3:4b-instruct"     # ollama model 

PROMPT_TEMPLATE = """
Answer the question based only on the following context:

{context}

---

Answer the question based on the above context: {question}
"""

- `CORPUS` (which sets `CHROMA_PATH`) and the embedding model must be the same as when the database was built. Otherwise the question's vector and the stored vectors cannot be compared.
- `K` is how many chunks go into the prompt. With 512-token chunks, 5 chunks is about 2,500 tokens of context.
- `PROMPT_TEMPLATE` has two placeholders, `{context}` and `{question}`. The instruction "based only on the context" keeps the model from answering from its own memory.

## Connect to the database and the LLM

In [ ]:
db = Chroma(persist_directory=CHROMA_PATH, embedding_function=get_embedding_function())
print(f"Chunks in database: {len(db.get(include=[])['ids'])}")

llm = ChatOllama(
    model=LLM_MODEL,
    reasoning=False,    # qwen3 thinks by default; this turns it off so it cannot loop in <think>
    num_ctx=8192,       # context window: room for K chunks + the answer
    num_predict=512,    # hard cap on answer length, so it cannot run forever 
    temperature=0.2,    # low = sticks closer to the context
)

- reasoning/thinking models appear to reach a num_predict cap quickly, before they can output an answer

This is done once, outside the query function, so each query does not reopen the database or reload settings.

The printed count should equal the number of chunks you added (180 for the 512-token run). If it is 0, the database is empty or `CHROMA_PATH` points to the wrong folder.

The LLM settings address the Qwen3 problems you hit before: `reasoning=False` stops the endless thinking, `num_ctx` leaves room for the prompt, and `num_predict` caps the output. They are not tested on your models. If you switch to a model that is not a reasoning model, you can remove `reasoning=False`.

## Step 1: retrieval only test - chunk inspection 

In [ ]:
def retrieve(query_text: str, k: int = K):
    # Embeds the question and returns the k nearest chunks as (Document, score) pairs.
    # The score is a distance: LOWER means closer.
    return db.similarity_search_with_score(query_text, k=k) # can define a minimal score_threshold if results appear dissimilar.


def show_results(results):
    for rank, (doc, score) in enumerate(results, start=1):
        m = doc.metadata
        print(f"[{rank}] score={score:.3f}  page={m.get('page')}  file={m.get('source', '').split('/')[-1]}")
        print(f"    section: {m.get('headings')}")
        print(f"    id: {m.get('id')}")
        print(f"    text: {doc.page_content}...\n")


question = "What cells drive liver fibrosis in MASH?"
results = retrieve(question)
show_results(results)

This runs only the search, with no reasoning LLM. 
Check ifthe retrieved chunks are not relevant to the question, can diagnose if a bad answer is due to irrelevant chunk retrieval vs downstream issues with the reasoning LLM. 

For each result you see the distance score (lower is closer), the page, the PDF, the section heading and the start of the text. These come from the flat metadata saved by `flatten_metadata`. The tutorial could only show an ID.

## Step 2: the full pipeline

- By default, show_chunks and show_prompt are set to False, can manually set to True 

In [ ]:
def query_rag(query_text: str, k: int = K, show_prompt: bool = False, show_chunks: bool = False):
    results = retrieve(query_text, k=k)
    if show_chunks:
        show_results(results)  # each retrieved chunk with score, page, section, id and text

    # Join their text into one context block, separated by ---
    context_text = "\n\n---\n\n".join(doc.page_content for doc, _score in results)

    # Fill in prompt template 
    prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE).format(
        context=context_text, question=query_text
    )
    if show_prompt:
        print(prompt)

    # 4. Ask the LLM. ChatOllama returns a message object; .content is the text
    response_text = llm.invoke(prompt).content

    sources = [
        f"{doc.metadata.get('source', '').split('/')[-1]} (p.{doc.metadata.get('page')}) "
        f"- {doc.metadata.get('headings')}"
        for doc, _score in results
    ]
    print(f"Response: {response_text}\n")
    print("Sources:")
    for s in sources:
        print(f"  - {s}")
    return response_text

What each step does:
1. **Retrieve**: the same search as step 1.
2. **Join**: the chunk texts become one string, so the model sees all of them at once.
3. **Fill the prompt**: `{context}` and `{question}` are replaced. Set `show_prompt=True` to print exactly what the LLM receives.
4. **Ask the LLM**: `invoke` sends the prompt to Ollama. `ChatOllama` returns a message object, so the code takes `.content`. The tutorial's `Ollama` returned a plain string.
5. **Sources**: the tutorial listed the chunk IDs. This version shows file, page and section heading, which is easier to read and check.

## Ask a question

In [ ]:
query_rag("What cells drive liver fibrosis in MASH?")

Run this after step 1 looks sensible. If answers are poor, check in this order:
1. Are the retrieved chunks relevant (step 1)? If not, the problem is chunking, embeddings or `k`, not the LLM.
2. Print the prompt with `show_prompt=True` and check the context is what you expect.
3. Only then try a different LLM or different settings.

! CURRENT ISSUES - with qwen3 4b thinking

- Current model (qwen3:4b) never reached a final answer. Some kind of token limit? Continued reasoning. 

! Resolved by changing to a non thinking/reasoning model (qwen3:4b-instruct). For the prompt "What cells drive liver fibrosis in MASH?", used 107 tokens.